# Aggregate Attack Strength Curve Probe Shards

本 notebook 只负责聚合已经落盘的 `attack_strength_curve_probe` shard results, 并可选重建阶段四论文图表。

它不会运行真实 VAE runner, 也不会新增 shard。

## User Config

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动克隆仓库使用的 HTTPS 地址。
REPO_BRANCH = "explicit-sync"  # 需要验证的分支名称。
DRIVE_ROOT = "/content/drive/MyDrive"  # Google Drive 挂载后的根目录。
RESULT_ROOT = f"{DRIVE_ROOT}/TSTW/results"  # TSTW 受治理结果根目录。
LOCAL_ROOT = "/content/TSTW_runtime"  # Colab session-local 工作区。
REPO_DIR = f"{LOCAL_ROOT}/repo"  # 仓库克隆到 Colab 本地后的路径。

RECORD_SOURCE_MODE = "internal_sweep"  # 聚合来源: internal_sweep, base_records, all。
REQUIRED_SHARD_SHORT_COMMIT = ""  # 可填 shard 短 commit; 留空表示自动选择最新完整 shard group。
REQUIRED_SHARD_COUNT = 1  # A100-40G 单 shard 全量默认要求 1 个 shard; 聚合前应确认最新 sc01 是全量而非 smoke。
TARGET_FPR = 0.01  # 聚合表使用的固定目标 FPR。

RUN_REPOSITORY_SMOKE_TESTS = True  # 是否先运行攻击强度曲线相关轻量测试。
RUN_ATTACK_STRENGTH_AGGREGATION = True  # 是否聚合 shard records。
RUN_PAPER_ARTIFACT_REBUILD = True  # 是否重建 paper_artifact_gate 以生成论文图表。
PAPER_SKIP_FIGURES = False  # 是否跳过 PNG/PDF 图表导出。
OVERWRITE_EXISTING_OUTPUT = False  # 聚合输出通常带 UTC 时间, 一般不需要覆盖。


## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Prepare Local Workspace And Repository

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command, cwd=None, check=True):
    """运行命令并实时打印 stdout/stderr。"""
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if check and completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed

local_root = Path(LOCAL_ROOT)
if local_root.exists():
    shutil.rmtree(local_root)
local_root.mkdir(parents=True, exist_ok=True)

run_command(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR])
short_commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).stdout.strip()
print(json.dumps({"repo_dir": REPO_DIR, "short_commit": short_commit}, ensure_ascii=False, indent=2))


## Verify Repository Contract

In [ ]:
if RUN_REPOSITORY_SMOKE_TESTS:
    run_command([
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/constraints/test_attack_strength_curve_probe_builder.py",
        "tests/constraints/test_paper_artifact_gate_builder.py",
    ], cwd=REPO_DIR)
else:
    print("已跳过仓库轻量约束测试。")


## Resolve Shard Group

该步骤只做 dry-run 式路径解析, 用于确认聚合的是同一 short commit 和同一 shard_count 的完整 shard group。

In [ ]:
# 通过聚合脚本的错误和最终 summary 检查完整性; 此处先打印候选目录, 便于人工确认。
shard_root = Path(RESULT_ROOT) / "attack_strength_curve_probe" / "shard_runs"
if not shard_root.exists():
    raise FileNotFoundError(shard_root)
candidate_dirs = sorted([p for p in shard_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
print(json.dumps([p.name for p in candidate_dirs[:30]], ensure_ascii=False, indent=2))


## Aggregate Attack Strength Curve

In [ ]:
attack_strength_output = None
if RUN_ATTACK_STRENGTH_AGGREGATION:
    aggregation_args = [
        "scripts/package_results/build_attack_strength_curve_probe.py",
        "--result-root", RESULT_ROOT,
        "--target-fpr", str(TARGET_FPR),
        "--short-commit", short_commit,
        "--record-source-mode", RECORD_SOURCE_MODE,
    ]
    if REQUIRED_SHARD_SHORT_COMMIT:
        aggregation_args.extend(["--required-shard-short-commit", REQUIRED_SHARD_SHORT_COMMIT])
    if REQUIRED_SHARD_COUNT is not None:
        aggregation_args.extend(["--required-shard-count", str(REQUIRED_SHARD_COUNT)])
    if OVERWRITE_EXISTING_OUTPUT:
        aggregation_args.append("--overwrite")
    completed = run_command([sys.executable, *aggregation_args], cwd=REPO_DIR)
    attack_strength_output = json.loads(completed.stdout)
else:
    print("已跳过攻击强度曲线聚合。")


## Check Aggregated Output

In [ ]:
if attack_strength_output is None:
    raise RuntimeError("attack_strength_output 为空, 当前 notebook 没有产生聚合结果。")
agg_root = Path(attack_strength_output["output_root"])
required_paths = {
    "records": agg_root / "records" / "attack_strength_event_scores.jsonl",
    "tpr_table": agg_root / "tables" / "attack_strength_tpr_table.csv",
    "auc_table": agg_root / "tables" / "attack_strength_auc_table.csv",
    "figure_data": agg_root / "figure_data" / "attack_strength_curve_figure_data.csv",
    "manifest": agg_root / "artifacts" / "attack_strength_curve_manifest.json",
}
manifest = json.loads(required_paths["manifest"].read_text(encoding="utf-8")) if required_paths["manifest"].exists() else {}
completion_summary = {
    "result_flow": "attack_strength_shard_aggregate",
    "output_root": str(agg_root),
    "required_paths": {key: path.exists() for key, path in required_paths.items()},
    "record_count": manifest.get("record_count"),
    "tpr_row_count": manifest.get("tpr_row_count"),
    "auc_row_count": manifest.get("auc_row_count"),
    "source_modes": manifest.get("source_modes"),
    "claim_support_allowed": manifest.get("claim_support_allowed"),
    "record_path_resolution": attack_strength_output.get("summary", {}).get("record_path_resolution"),
}
completion_summary["status"] = all(completion_summary["required_paths"].values()) and bool(manifest.get("record_count"))
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)


## Rebuild Paper Artifacts

In [ ]:
paper_artifact_output = None
if RUN_PAPER_ARTIFACT_REBUILD:
    paper_args = [
        "scripts/package_results/build_paper_artifact_gate.py",
        "--result-root", RESULT_ROOT,
        "--short-commit", short_commit,
    ]
    if PAPER_SKIP_FIGURES:
        paper_args.append("--skip-figures")
    completed = run_command([sys.executable, *paper_args], cwd=REPO_DIR)
    paper_artifact_output = json.loads(completed.stdout)
else:
    print("已跳过 paper_artifact_gate 重建。")


## Final Summary

In [ ]:
final_summary = {
    "short_commit": short_commit,
    "attack_strength_output": attack_strength_output,
    "paper_artifact_output": paper_artifact_output,
}
print(json.dumps(final_summary, ensure_ascii=False, indent=2))
